In [45]:
from sklearn.model_selection import train_test_split
import pandas as pd
import torch

In [46]:
df=pd.read_csv("/content/fashion-mnist_train.csv")
df.head()
torch.manual_seed(42)

In [47]:
df.shape

(20821, 785)

In [48]:
x=df.iloc[:,1:].values
y=df.iloc[:,0].values
y

array([2, 9, 6, ..., 0, 8, 3])

In [49]:
x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=42)

In [50]:
from torchvision import transforms

coustom_transform=transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(254),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])

In [51]:
from torch.utils.data import DataLoader,Dataset

In [52]:
from PIL import Image
import numpy as np

class MyClass(Dataset):

  def __init__(self,features,labels,transform):
    self.features=features
    self.labels=labels
    self.transform=transform

  def __len__(self):
    return len(self.features)

  def __getitem__(self,index):
    # reshape in 28*28
    image=self.features[index].reshape(28,28)

    # convert into uint data type
    image=image.astype(np.uint8)

    # convert into 3 channels means in coloured
    image=np.stack([image]*3,axis=-1)

    # convert into PIL image
    image=Image.fromarray(image)

    # apply transformations
    image=self.transform(image)

    return image,torch.tensor(self.labels[index],dtype=torch.long)








In [53]:
train_data=MyClass(x_train,y_train,transform=coustom_transform)
test_data=MyClass(x_test,y_test,transform=coustom_transform)
len(train_data),len(test_data)

(15615, 5206)

In [54]:
train_loader=DataLoader(train_data,batch_size=32,shuffle=True,pin_memory=True)
test_loader=DataLoader(test_data,batch_size=32,shuffle=False,pin_memory=True)
len(train_loader),len(test_loader)

(488, 163)

In [55]:
# define pretrained model
import torchvision.models as models

model=models.vgg16(pretrained=True)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [56]:
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [57]:
model.features
#  frizzing parameters

for param in model.features.parameters():
  param.requires_grad=False


In [58]:
model.classifier
# replacing cgg16 classifier to our classifier
model.classifier=torch.nn.Sequential(
    torch.nn.Linear(25088,1024),
    torch.nn.ReLU(),
    torch.nn.Dropout(p=0.3),
    torch.nn.Linear(1024,512),
    torch.nn.ReLU(),
    torch.nn.Dropout(p=0.3),
    torch.nn.Linear(512,10)
)

In [59]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=model.to(device)

In [77]:
epoches=8
lr=0.0001
optimizer=torch.optim.Adam(model.classifier.parameters(),lr=lr)
criteria=torch.nn.CrossEntropyLoss()

In [ ]:
i=0
for epoch  in range(epoches):
  total_loss=0
  for batch_features,batch_labels in train_loader:
    batch_features=batch_features.to(device)
    batch_labels=batch_labels.to(device)

    y_pred=model(batch_features)

    loss=criteria(y_pred,batch_labels)

    optimizer.zero_grad()
    loss.backward()

    optimizer.step()

    total_loss += loss.item()
    i +=1
  avg_loss= total_loss/len(train_loader)

  print(f"in {i}th loop loss is {avg_loss}")

/tmp/ipykernel_2832/1363959196.py:19: RuntimeWarning: invalid value encountered in cast
  image=image.astype(np.uint8)


in 488th loop loss is 0.033638307909746264
in 976th loop loss is 0.0165087615121846
in 1464th loop loss is 0.004920979834815014
in 1952th loop loss is 0.008104313184766338
in 2440th loop loss is 0.001076389653116032
in 2928th loop loss is 0.00016312473066240027
in 3416th loop loss is 6.0276833155102557e-05
in 3904th loop loss is 3.429027766869098e-05


In [70]:
model.eval()
correct=0
total=0
for batch_features,batch_labels in test_loader:
  batch_features=batch_features.to(device)
  batch_labels=batch_labels.to(device)

  y_pred=model(batch_features)

  _, predicted = torch.max(y_pred, 1)

  total += batch_labels.size(0)
  correct += (predicted == batch_labels).sum().item()
print(f"test accuracy is {(correct/total)*100}%")

test accuracy is 90.33807145601229%


In [76]:
correct=0
total=0

for batch_features , batch_labels in train_loader:
  batch_features=batch_features.to(device)
  batch_labels=batch_labels.to(device)

  y_pred=model(batch_features)

  _,predicted=torch.max(y_pred,1)

  total += batch_labels.size(0)
  correct += (predicted==batch_labels).sum().item()
print(f"train accuracy is {(correct/total)*100}%")

/tmp/ipykernel_2832/1363959196.py:19: RuntimeWarning: invalid value encountered in cast
  image=image.astype(np.uint8)


train accuracy is 99.39161063080371%
